### Preparacion para Google Colab

Correr una sola vez al empezar.

In [ ]:
%pip install --quiet astroNN umap-learn
from astroNN.datasets import load_galaxy10sdss
_ = load_galaxy10sdss()
print('Listo.')

# Practica 4: el espacio latente de las galaxias

Curso: Inteligencia Artificial para Astrofisica (introductorio)

Las tres clases de hoy en un solo notebook: la **CNN** como encoder, la
**representacion** que se aprende sin etiquetas, y la **fisica que le
imponemos** al modelo.

Vamos a construir, paso a paso, representaciones cada vez mejores del mismo
conjunto de galaxias. Los numeros del recorrido quedan para el final.

**Como se trabaja:** el andamiaje esta escrito. Hay que completar donde dice
`# >>> COMPLETAR <<<`, correr todo de arriba hacia abajo, y mirar los graficos.
La practica entera corre en unos 5 minutos.

**CodeX:** no se usa hasta la Parte 5, que esta marcada. Antes de eso, se
puede consultar la documentacion de PyTorch.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

DIM_LATENTE = 8      # cuantos numeros describen a cada galaxia
EPOCAS = 25

## Antes de empezar: los datos

Esta celda mira si el dataset esta disponible. No descarga nada y no rompe:
si no encuentra las galaxias reales, sigue con un set de demostracion para que
el notebook corra igual.

In [ ]:
import importlib.util

# Buscamos el archivo en esta carpeta y en las de mas arriba, asi funciona
# igual si el notebook esta en una subcarpeta.
CANDIDATOS = ["Galaxy10.h5", "../Galaxy10.h5", "../../Galaxy10.h5",
              os.path.expanduser("~/Galaxy10.h5")]
ARCHIVO = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if ARCHIVO:
    MODO = "archivo local"
    print("Encontre las galaxias reales en:", os.path.abspath(ARCHIVO))
elif importlib.util.find_spec("astroNN") is not None:
    MODO = "descarga astroNN"
    print("No esta Galaxy10.h5, pero si astroNN.")
    print("La celda que sigue va a descargar ~210 MB la primera vez.")
else:
    MODO = "demostracion"
    print("=" * 68)
    print("NO ENCONTRE LAS GALAXIAS REALES. Sigo con un set de DEMOSTRACION.")
    print("Todo el notebook corre igual, pero las imagenes son sinteticas.")
    print()
    print("Para trabajar con los datos reales, cualquiera de estas dos:")
    print("  1) copiar Galaxy10.h5 (el que se comparte en clase) a")
    print("     " + os.getcwd())
    print("     (o a la carpeta de arriba: tambien lo busca ahi)")
    print("  2) instalar astroNN:  pip install astroNN")
    print("     (en Colab:  !pip install astroNN )")
    print()
    print("Despues, correr el notebook de nuevo desde arriba.")
    print("=" * 68)

## Los datos

Galaxias reales del relevamiento SDSS (el dataset **Galaxy10**), agrupadas
en tres morfologias: lisas, disco de canto y espirales. Trabajamos **en color**: las
tres bandas traen fisica de poblaciones estelares (elipticas rojas, espirales
azules) y no hay razon para tirarla. Bajamos a 32x32 para que corra rapido;
el `mode="area"` promedia los pixeles de cada celda, o sea que conserva el
flujo (el modo por defecto se queda con un pixel de cada dos y tira el resto).

Lo importante: **las etiquetas se cargan pero no se usan para entrenar.**
Quedan guardadas para colorear los graficos y para medir, al final, si el
modelo llego a esa estructura por su cuenta.

El split va **antes de entrenar**: 80% entrenamiento, 20% validacion, y la
validacion no se toca. ¿Y el test? Aca no hace falta: el test existe para
cuando uno **elige** un modelo mirando la validacion (Practica 1). Aca no
elegimos nada; la validacion solo mide.

In [ ]:
def galaxias_de_demostracion(n_por_clase=1200, lado=69):
    "Galaxias sinteticas: elipticas, de canto y espirales. Solo para que corra."
    r = np.random.default_rng(0)
    yy, xx = np.mgrid[0:lado, 0:lado] - (lado - 1) / 2.0
    ims, ys = [], []
    for c in range(3):
        for _ in range(n_por_clase):
            a = r.uniform(0, np.pi); ca, sa = np.cos(a), np.sin(a)
            xr, yr = xx * ca + yy * sa, -xx * sa + yy * ca
            if c == 0:
                q, re = r.uniform(0.75, 1.0), r.uniform(9.0, 14.0)
                rr = np.sqrt(xr ** 2 + (yr / q) ** 2); img = np.exp(-(rr / re) ** 0.8)
            elif c == 1:
                q, re = r.uniform(0.14, 0.25), r.uniform(13.0, 18.0)
                rr = np.sqrt(xr ** 2 + (yr / q) ** 2); img = np.exp(-(rr / re) ** 0.9)
            else:
                q, re = r.uniform(0.7, 1.0), r.uniform(6.0, 9.0)
                rr = np.sqrt(xr ** 2 + (yr / q) ** 2); img = 0.55 * np.exp(-(rr / re) ** 0.8)
                th = np.arctan2(yr / q, xr)
                brazos = np.cos(2 * (th - np.log(rr + 1.2) * r.uniform(2.4, 3.4)))
                img = img + 0.8 * np.exp(-((rr - 15.0) ** 2) / 110.0) * np.clip(brazos, 0, 1)
            img = np.clip(img + r.normal(0, 0.03, img.shape), 0, None)
            ims.append(np.repeat((img / img.max() * 255)[:, :, None], 3, axis=2))
            ys.append(c)
    idx = r.permutation(len(ys))
    return np.array(ims)[idx].astype(np.uint8), np.array(ys)[idx]


if MODO == "demostracion":
    images, labels3 = galaxias_de_demostracion()
else:
    if MODO == "archivo local":
        import h5py
        with h5py.File(ARCHIVO, "r") as f:
            images = np.array(f["images"])
            labels = np.array(f["ans"])
    else:
        from astroNN.datasets import load_galaxy10sdss
        images, labels = load_galaxy10sdss()

    grupos = {1: 0, 2: 0, 3: 0, 4: 1, 5: 1, 6: 1, 7: 2, 8: 2, 9: 2}
    mascara = np.isin(labels, list(grupos))
    images, labels = images[mascara], labels[mascara]
    labels3 = np.array([grupos[l] for l in labels])

    # submuestreo balanceado: 1200 por clase
    idx = np.concatenate([
        np.random.permutation(np.where(labels3 == c)[0])[:1200] for c in range(3)
    ])
    np.random.shuffle(idx)
    images, labels3 = images[idx], labels3[idx]

X = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
X = F.interpolate(X, size=32, mode="area")       # conserva el flujo
# Una sola escala por galaxia, comun a las tres bandas: asi el brillo queda
# normalizado pero los cocientes entre bandas (el color) se preservan.
X = X / X.amax(dim=(1, 2, 3), keepdim=True).clamp(min=1e-6)
y = torch.tensor(labels3, dtype=torch.int64)

n_train = int(0.8 * len(X))
X_train, y_train = X[:n_train], y[:n_train]
X_val,   y_val   = X[n_train:], y[n_train:]

nombres_clases = ["lisa", "disco de canto", "espiral"]
print("modo:", MODO)
print("entrenamiento:", tuple(X_train.shape), " validacion:", tuple(X_val.shape))

In [ ]:
# Como se ven (esta celda viene dada)
fig, axs = plt.subplots(3, 8, figsize=(9, 3.6))
for c in range(3):
    ejemplos = torch.where(y_train == c)[0][:8]
    for j, i in enumerate(ejemplos):
        axs[c, j].imshow(X_train[i].permute(1, 2, 0).clamp(0, 1))
        axs[c, j].set_xticks([]); axs[c, j].set_yticks([])
    axs[c, 0].set_ylabel(nombres_clases[c], fontsize=9)
fig.suptitle("Galaxy10, en color y a 32x32")
plt.tight_layout(); plt.show()

## Parte 1: el encoder

El encoder es la misma idea que la CNN de la Practica 2: tres
convoluciones que suben los canales (16 → 32 → 64) mientras bajan la
resolucion. Dos diferencias, y las dos son decisiones con motivo:

- **Achicamos con `stride=2` en la convolucion, sin MaxPool.** Es otra manera
  de resumir el mapa (la convolucion salta de a dos pixeles); mismo rol,
  una capa menos.
- **No hay `MaximoGlobal`.** En la Practica 2 tiramos el *donde* porque para
  clasificar no hacia falta. Aca el codigo tiene que alcanzarle al decoder
  para **redibujar la galaxia entera**, y para eso el donde es imprescindible:
  el encoder termina en `Flatten` + `Linear`, que lo conserva. Ojo con esto,
  porque va a volver: el donde que conservamos incluye la orientacion, y que
  hacer con ella es el tema de la Parte 4.

En lugar de terminar en probabilidades de clase, termina en `DIM_LATENTE`
numeros que no significan nada a priori.

Completar la lista de capas. La receta:

| capa | entrada | salida |
|---|---|---|
| `Conv2d(3, 16, 3, stride=2, padding=1)` + `ReLU` | 3x32x32 | 16x16x16 |
| `Conv2d(16, 32, 3, stride=2, padding=1)` + `ReLU` | 16x16x16 | 32x8x8 |
| `Conv2d(32, 64, 3, stride=2, padding=1)` + `ReLU` | 32x8x8 | 64x4x4 |
| `Flatten` | 64x4x4 | 1024 |
| `Linear(1024, DIM_LATENTE)` | 1024 | 8 |

Sin activacion en la ultima capa: el codigo puede ser positivo o negativo.

In [ ]:
def hacer_encoder():
    return nn.Sequential(
        # >>> COMPLETAR <<<  (los tres bloques Conv2d+ReLU, Flatten y Linear)
    )

prueba = hacer_encoder()(X_train[:5])
print("codigo de 5 galaxias:", tuple(prueba.shape))   # tiene que dar (5, 8)

El decoder hace el camino inverso con `ConvTranspose2d` y viene dado.
La clase `Autoencoder` junta los dos y devuelve la reconstruccion y el codigo.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = hacer_encoder()
        self.expandir = nn.Linear(DIM_LATENTE, 64 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 4, stride=2, padding=1), nn.Sigmoid(),
        )

    def codigo(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.codigo(x)
        h = self.expandir(z).view(-1, 64, 4, 4)
        return self.decoder(h), z


modelo_prueba = Autoencoder()
n_par = sum(p.numel() for p in modelo_prueba.parameters())
print(f"parametros del autoencoder: {n_par:,}")

## Parte 2: entrenar sin etiquetas

La perdida es solo `MSE(reconstruccion, entrada)`. No aparece `y` en ningun
lado: el modelo nunca ve una etiqueta.

`rotar` gira el lote un angulo cualquiera (los multiplos de 90 grados salen
exactos, sin interpolar; el resto usa `grid_sample`). La vamos a usar a partir
de la Parte 4.

In [ ]:
def rotar(x, grados):
    "Rota el lote un angulo cualquiera, en grados, alrededor del centro."
    if grados % 90 == 0:                       # exacto, sin interpolar
        return torch.rot90(x, int(grados // 90) % 4, dims=(2, 3))
    th = torch.tensor(float(grados) * np.pi / 180.0)
    cero = torch.tensor(0.0)
    R = torch.stack([torch.stack([torch.cos(th), -torch.sin(th), cero]),
                     torch.stack([torch.sin(th),  torch.cos(th), cero])])
    grilla = F.affine_grid(R.expand(len(x), 2, 3), list(x.shape), align_corners=False)
    return F.grid_sample(x, grilla, align_corners=False)


def entrenar(modelo, epocas=EPOCAS, lr=2e-3, titulo=""):
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    hist_train, hist_val = [], []
    for ep in range(epocas):
        perm = torch.randperm(len(X_train))
        total = 0.0
        for i in range(0, len(X_train), 128):
            xb = X_train[perm[i:i + 128]]
            opt.zero_grad()
            xhat, z = modelo(xb)
            perdida = ((xhat - xb) ** 2).mean()
            perdida.backward(); opt.step()
            total += float(perdida) * len(xb)
        hist_train.append(total / len(X_train))
        with torch.no_grad():                        # misma medida, datos nunca vistos
            xv, _ = modelo(X_val)
            hist_val.append(float(((xv - X_val) ** 2).mean()))
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"  epoca {ep + 1:2d}/{epocas}   train {hist_train[-1]:.5f}"
                  f"   val {hist_val[-1]:.5f}")
    ejes = range(1, epocas + 1)
    fig, ax = plt.subplots(figsize=(5.6, 3.2))
    ax.semilogy(ejes, hist_train, color="#424242", lw=1.8, label="entrenamiento")
    ax.semilogy(ejes, hist_val, color="#c74a34", lw=1.8, label="validacion")
    ax.set_xlabel("epoca"); ax.set_ylabel("reconstruccion (MSE)")
    ax.set_title(titulo or "entrenamiento", fontsize=11)
    ax.legend(frameon=False, fontsize=9)
    for lado in ax.spines.values():
        lado.set_color("#cccccc")
    plt.tight_layout(); plt.show()
    return hist_train


def ver_reconstrucciones(modelo, titulo=""):
    with torch.no_grad():
        xhat, _ = modelo(X_val[:8])
    fig, axs = plt.subplots(2, 8, figsize=(9, 2.6))
    for j in range(8):
        axs[0, j].imshow(X_val[j].permute(1, 2, 0).clamp(0, 1))
        axs[1, j].imshow(xhat[j].permute(1, 2, 0).clamp(0, 1))
        for r in range(2):
            axs[r, j].set_xticks([]); axs[r, j].set_yticks([])
    axs[0, 0].set_ylabel("original", fontsize=9)
    axs[1, 0].set_ylabel("reconstruida", fontsize=9)
    fig.suptitle(titulo or "reconstruccion")
    plt.tight_layout(); plt.show()

Ahora entrenar el autoencoder base y mirar las reconstrucciones.
Tarda menos de un minuto. La sintaxis, para copiar y adaptar:

```python
historia = entrenar(modelo, titulo="un titulo")
ver_reconstrucciones(modelo, "un titulo")
```

In [ ]:
ae_base = Autoencoder()
# >>> COMPLETAR <<<  (llamar a entrenar y despues a ver_reconstrucciones)

## Parte 3: mirar y medir el espacio latente

Cada galaxia de validacion es ahora un punto en 8 dimensiones. Para verlo en
el papel lo proyectamos a 2 con PCA (las dos direcciones donde los codigos mas
se dispersan).

**Recien aca aparecen las etiquetas, y solo para pintar los puntos.**
Si los colores quedan separados, esa separacion la encontro el modelo solo.

Para mirar usamos **UMAP** si esta instalado (`pip install umap-learn`;
en Colab ya viene), que conserva vecindades: puntos cercanos en las 8
dimensiones quedan cercanos en el dibujo. Si no esta, se usa PCA, que es lineal
y aplasta mas. En cualquiera de los dos casos el dibujo pierde estructura, las
distancias entre grupos no significan nada, y por eso **no medimos sobre el
dibujo**. Medimos con la **sonda lineal**, calculada en las 8 dimensiones
completas: una regresion logistica entrenada sobre el codigo para predecir la
clase, con validacion cruzada de 5. Es deliberadamente tonta: si un modelo
lineal acierta, es porque el autoencoder ya dejo las clases casi separadas.
Azar = 0.333.

In [ ]:
def codigos(modelo, X):
    with torch.no_grad():
        return modelo.codigo(X).numpy()


def proyectar(Z):
    "A 2 dimensiones: UMAP si esta instalado, si no PCA. Devuelve (P, metodo)."
    Zn = (Z - Z.mean(0)) / (Z.std(0) + 1e-8)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            import umap
            P = umap.UMAP(n_neighbors=30, min_dist=0.1,
                          random_state=0).fit_transform(Zn)
        return P, "UMAP"
    except ImportError:
        U, S, Vt = np.linalg.svd(Zn, full_matrices=False)
        return Zn @ Vt[:2].T, "PCA"


def dibujar_latente(ax, P, y, titulo="", maximo=300):
    "Dibuja hasta `maximo` puntos (sorteados fijo) para que el ojo respire."
    sel_n = np.random.default_rng(0).permutation(len(P))[:maximo]
    P, yy = P[sel_n], y.numpy()[sel_n]
    for c, m in zip(range(3), ["o", "s", "^"]):
        sel = (yy == c)
        ax.scatter(P[sel, 0], P[sel, 1], s=22, marker=m, alpha=0.8,
                   edgecolors="white", linewidths=0.4,
                   label=nombres_clases[c])
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(titulo, fontsize=10)


def graficar_latente(Z, y, titulo=""):
    P, metodo = proyectar(Z)
    fig, ax = plt.subplots(figsize=(5.0, 4.0))
    dibujar_latente(ax, P, y, f"{titulo or 'espacio latente'} ({metodo})")
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


def sonda_lineal(Z, y):
    "Acierto de un clasificador lineal sobre el codigo. Azar = 1/3."
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    return float(cross_val_score(LogisticRegression(max_iter=3000),
                                 Z, y.numpy(), cv=5).mean())

In [ ]:
# La sintaxis de las tres funciones, para copiar y adaptar:
#   Z = codigos(modelo, X_val)
#   graficar_latente(Z, y_val, "un titulo")
#   s = sonda_lineal(Z, y_val)
# >>> COMPLETAR <<<  (calcular los codigos de validacion en Z_base,
#                     graficarlos con graficar_latente, y medir sonda_base)

## Parte 4: la galaxia rotada

Una galaxia rotada sigue siendo la misma galaxia: su orientacion en el cielo
es azar de proyeccion, no una propiedad del objeto. ¿Nuestro modelo lo sabe?

La celda de abajo (viene dada) lo muestra: toma tres galaxias, las rota 37
grados, y marca en el mapa donde cae cada una (**estrella llena**) y donde
cae su version rotada (**estrella vacia**).

In [ ]:
def mapa_con_gemelas(modelo, titulo="", ids=(15, 80, 200), grados=37.0):
    "El mapa del latente, con tres galaxias (estrella llena) y sus rotadas (vacia)."
    Z = codigos(modelo, X_val)
    Zrot = codigos(modelo, rotar(X_val[list(ids)], grados))
    P, metodo = proyectar(np.vstack([Z, Zrot]))
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    dibujar_latente(ax, P[:len(Z)], y_val, f"{titulo} ({metodo})")
    for k in range(len(ids)):
        a, b = P[list(ids)[k]], P[len(Z) + k]
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", color="k", lw=1.6))
        ax.plot(*a, marker="*", ms=17, color="k")
        ax.plot(*b, marker="*", ms=17, mfc="none", mec="k", mew=1.8)
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


mapa_con_gemelas(ae_base, "autoencoder base")

Dos cosas para ver en ese mapa, y conviene decirlas juntas:

- Los grumos chicos son **puros**: adentro de cada uno hay una sola clase.
  Localmente, el base separa muy bien. Hasta aca parece el mejor espacio
  posible.
- Pero la estrella vacia cae **lejos** de la llena: la misma galaxia, apenas
  rotada, vive en otro lugar del espacio. Como buscador de un catalogo, este
  espacio esta roto: le pedis los vecinos de una galaxia y no te devuelve ni
  a la misma galaxia rotada.

El numero que resume esto es el **desalineo**: cuanto se mueve el codigo al
rotar, en unidades del tamaño del espacio latente. Cero es invariancia
perfecta; orden 1 significa que la rotada queda tan lejos como una galaxia
distinta.

In [ ]:
def desalineo(modelo, angulos=(37, 113, 216), X=None):
    "Cuanto se mueve el codigo al rotar, promediado sobre varios angulos."
    X = X_val if X is None else X
    with torch.no_grad():
        z0 = modelo.codigo(X)
        escala = z0.std(0).norm()
        d = [float((z0 - modelo.codigo(rotar(X, a))).pow(2).sum(1).sqrt().mean() / escala)
             for a in angulos]
    return float(np.mean(d))


des_base = desalineo(ae_base)
print(f"desalineo del autoencoder base: {des_base:.3f}")

¿Como se arregla? Hay tres caminos: en los **datos** (entrenar con imagenes
rotadas al azar), en la **perdida** (un termino que penalice que el codigo
cambie al rotar; eso es exactamente lo que hace la PINN de la Parte 8 con su
ecuacion), o en la **arquitectura** (un encoder que no *pueda* cambiar al
rotar). Vamos directo al mas fuerte: la arquitectura.

## Parte 5: **aca si, CodeX**

Falta la tercera manera: hacer el encoder invariante **por construccion**.
La idea, en una linea: codificar la imagen en varias rotaciones y promediar los
codigos. Si rotamos la entrada por uno de esos mismos angulos, las rotaciones
son las mismas en otro orden, y el promedio no cambia.

Pedirle a CodeX que escriba una clase `AutoencoderInvariante` que herede de
`Autoencoder` y redefina `codigo` de esa manera, con la cantidad de angulos
como parametro (`n_angulos`, repartidos parejo en 360 grados). Despues
entrenarla, medir su desalineo y su sonda, y mirar su mapa con
`mapa_con_gemelas`.

**Reglas:** prompts cortos, todo se corre aca, y si una linea no se entiende
hay que pedirle que la explique. Los prompts van en la celda de abajo.

In [ ]:
# Con CodeX. Al final tienen que quedar definidas estas variables:
#   ae_inv, des_inv, sonda_inv, Z_inv
# y mostrar el mapa con:  mapa_con_gemelas(ae_inv, "invariante")

Dos cosas para mirar antes de seguir:

- En el mapa, la estrella vacia ahora cae **encima** de la llena: el buscador
  de catalogo quedo arreglado.
- A multiplos de 90 grados el desalineo da **cero exacto**: rotar la entrada
  permuta los cuatro terminos del promedio, y un promedio no depende del
  orden. A un angulo cualquiera no es cero: la garantia vale sobre el grupo
  que impusimos (4 rotaciones); la simetria del cielo es continua.

In [ ]:
# Prompts que usaste con CodeX (una linea cada uno):
prompts = [
    "...",
]
for p in prompts:
    print("-", p)

## Parte 6: cambiar el objetivo (contrastivo)

Hasta aca todo salio de la misma pérdida: reconstruir pixeles. Y la
reconstruccion paga lo que domina el error, brillo, tamaño, orientacion; los
brazos espirales debiles valen centavos. **El modelo aprende lo que la perdida
paga.**

El aprendizaje contrastivo tira la reconstruccion a la basura y pide
directamente lo que queremos: que **dos vistas del mismo objeto caigan juntas**
en el espacio latente, y que objetos distintos caigan lejos. Una "vista" es la
galaxia rotada un angulo al azar (y espejada a veces): la augmentation, que en
el autoencoder no servia, aca *define* que significa "mismo objeto".

La perdida es InfoNCE. Para un lote de $N$ galaxias con vistas $u_i$ y $v_i$
(normalizadas a la esfera unidad):

$$\mathcal{L} = -\frac{1}{N}\sum_i \log
  \frac{e^{u_i \cdot v_i / \tau}}{\sum_j e^{u_i \cdot v_j / \tau}}$$

Es una clasificacion de $N$ clases: la vista $u_i$ tiene que reconocer a su
pareja $v_i$ entre las $N$ candidatas. En PyTorch eso es `cross_entropy` sobre
la matriz de similitudes, con la diagonal como etiqueta.

Completar la perdida (esta indicado). Se puede pedir ayuda a CodeX; vale todo
lo de la Parte 5.

In [ ]:
def vista(xb):
    "Una vista del lote: rotacion continua al azar + espejo la mitad de las veces."
    xb = rotar(xb, float(np.random.uniform(0, 360)))
    if np.random.rand() < 0.5:
        xb = torch.flip(xb, dims=(3,))
    return xb


torch.manual_seed(0)
enc_con = hacer_encoder()
proyeccion = nn.Sequential(nn.ReLU(), nn.Linear(DIM_LATENTE, 32))
opt = torch.optim.Adam(list(enc_con.parameters()) + list(proyeccion.parameters()), lr=2e-3)
TAU = 0.2

for ep in range(EPOCAS):
    perm = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 128):
        xb = X_train[perm[i:i + 128]]
        if len(xb) < 8:
            continue
        opt.zero_grad()
        u = F.normalize(proyeccion(enc_con(vista(xb))), dim=1)
        v = F.normalize(proyeccion(enc_con(vista(xb))), dim=1)
        # >>> COMPLETAR <<<  la perdida InfoNCE:
        #   1) logits = matriz de similitudes u @ v.T, dividida por TAU
        #   2) la pareja correcta de u_i es v_i: etiquetas = arange(len(xb))
        #   3) perdida = cross_entropy(logits, etiquetas), promediada en las
        #      dos direcciones (logits y logits.T)
        perdida.backward(); opt.step()
    if (ep + 1) % 5 == 0:
        print(f"  epoca {ep + 1:2d}/{EPOCAS}   perdida {float(perdida):.3f}")

El codigo para medir es la salida del encoder (la proyeccion se usa solo para
entrenar y se descarta: es la receta estandar de SimCLR).

In [ ]:
class _Envoltura:
    "Para reusar codigos() y desalineo() con el encoder contrastivo."
    def codigo(self, x):
        return enc_con(x)

modelo_con = _Envoltura()
# >>> COMPLETAR <<<  (medir Z_con, des_con y sonda_con del modelo
#                     contrastivo, y graficar su latente)

Los tres espacios, lado a lado (esta celda viene dada). El titulo de cada
panel lleva su sonda: el orden visual y el orden numerico se comparan solos.

In [ ]:
panel = [("base", "Z_base", "sonda_base"),
         ("invariante", "Z_inv", "sonda_inv"),
         ("contrastivo", "Z_con", "sonda_con")]
panel = [(n, globals()[z], globals().get(s)) for n, z, s in panel
         if z in globals()]
if panel:
    fig, axs = plt.subplots(1, len(panel), figsize=(3.2 * len(panel), 3.4))
    axs = np.atleast_1d(axs)
    for ax, (nombre, Z, s) in zip(axs, panel):
        P, metodo = proyectar(Z)
        et = f"{nombre}" + (f"  (sonda {s:.2f})" if s is not None else "")
        dibujar_latente(ax, P, y_val, et)
    axs[0].legend(fontsize=8, loc="best")
    fig.suptitle(f"el mismo espacio de galaxias, tres objetivos ({metodo})")
    plt.tight_layout(); plt.show()
else:
    print("Faltan los codigos de las partes anteriores.")

## Parte 7: el techo

Ultima pregunta: ¿cuanto de la clase esta realmente en los pixeles? Para
saberlo entrenamos la misma CNN pero **con todas las etiquetas**, como
clasificador supervisado clasico. Ese numero es el techo: lo maximo que estos
datos dan.

Esta celda viene dada y tarda unos 15 segundos.

In [ ]:
torch.manual_seed(0)
cnn_sup = nn.Sequential(
    nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
    nn.Flatten(),
    nn.Linear(64 * 4 * 4, 3),
)
opt = torch.optim.Adam(cnn_sup.parameters(), lr=2e-3)
for ep in range(EPOCAS):
    perm = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 128):
        xb, yb = X_train[perm[i:i + 128]], y_train[perm[i:i + 128]]
        opt.zero_grad()
        F.cross_entropy(cnn_sup(xb), yb).backward()
        opt.step()
with torch.no_grad():
    techo = float((cnn_sup(X_val).argmax(1) == y_val).float().mean())
print(f"CNN supervisada (todas las etiquetas): {techo:.3f}")

**El arco completo, en los numeros que acabamos de construir:** un
clasificador al azar acierta 0.33. Un autoencoder que nunca vio una etiqueta
llega a ~0.60. Imponerle la simetria de rotacion lo sube a ~0.64. Cambiar el
objetivo por uno contrastivo lo sube a ~0.69. Y el techo, entrenando con todas
las etiquetas, es ~0.86. **Cada escalon es una decision de diseno nuestra, no
un cambio en los datos.**

La brecha entre el contrastivo y el techo no esta en los datos: esta en las
etiquetas, que el supervisado tiene y los demas no. Y la brecha entre el
autoencoder y el contrastivo tampoco esta en los datos: esta en el objetivo.
Mismos pixeles, misma arquitectura, tres representaciones distintas.
**La representacion la elige la perdida.**

## Parte 8: la misma maniobra, con una ecuacion

En la Parte 4 dijimos que una simetria tambien se puede imponer **en la
perdida**: un termino que penaliza que el codigo cambie al rotar. Una **PINN**
hace exactamente eso, pero lo que impone es una ecuacion diferencial.

Queremos la funcion $u(x)$ que cumple

$$\frac{du}{dx} + 2x\,u = 0, \qquad u(0) = 1$$

No tenemos datos de $u$: ni uno. Lo unico que tenemos es la ecuacion. La red
sale de ahi.

La perdida tiene dos terminos:

- el **residuo**: evaluamos $u' + 2xu$ en 200 puntos del intervalo y pedimos
  que de cero,
- la **condicion inicial**: $u(0) = 1$.

La derivada no se aproxima por diferencias finitas: sale de la misma red con
`torch.autograd.grad`, o sea que es exacta.

Corre en dos segundos y no necesita ninguna libreria nueva.

In [ ]:
def resolver_ode(usar_fisica=True, pasos=2000, semilla=0):
    "Resuelve u' + 2x u = 0 con u(0)=1. Si usar_fisica=False, solo impone u(0)=1."
    torch.manual_seed(semilla)
    u = nn.Sequential(nn.Linear(1, 32), nn.Tanh(),
                      nn.Linear(32, 32), nn.Tanh(),
                      nn.Linear(32, 1))
    opt = torch.optim.Adam(u.parameters(), lr=1e-2)
    x = torch.linspace(0, 3, 200).reshape(-1, 1)
    cero = torch.zeros(1, 1)

    for _ in range(pasos):
        opt.zero_grad()
        xg = x.clone().requires_grad_(True)
        ux = u(xg)
        du = torch.autograd.grad(ux.sum(), xg, create_graph=True)[0]
        residuo = du + 2 * xg * ux                 # <- la ecuacion

        perdida = (u(cero) - 1.0).pow(2).mean()    # condicion inicial
        if usar_fisica:
            perdida = perdida + residuo.pow(2).mean()
        perdida.backward()
        opt.step()

    with torch.no_grad():
        return x.numpy().ravel(), u(x).numpy().ravel(), float(perdida)


x_ode, u_pinn, perdida_pinn = resolver_ode(usar_fisica=True)
u_exacta = np.exp(-x_ode ** 2)
print(f"error maximo contra la solucion exacta: {np.abs(u_pinn - u_exacta).max():.4f}")
print(f"perdida final: {perdida_pinn:.2e}")
print()
print("Tiene que dar un error de orden 0.001. Si da mucho mas, subir `pasos`.")

Ahora el experimento: lo mismo **sin** el termino de la ecuacion, o sea pidiendo
unicamente que $u(0) = 1$. Esta celda ya corre las dos y muestra los numeros;
lo que falta completar es el grafico.

In [ ]:
x_ode, u_sin, perdida_sin = resolver_ode(usar_fisica=False)

print(f"CON la ecuacion:   error maximo {np.abs(u_pinn - u_exacta).max():.4f}"
      f"   perdida final {perdida_pinn:.2e}")
print(f"SIN la ecuacion:   error maximo {np.abs(u_sin - u_exacta).max():.4f}"
      f"   perdida final {perdida_sin:.2e}")

In [ ]:
# >>> COMPLETAR <<<  (graficar las tres curvas juntas: u_exacta, u_pinn y u_sin,
#                     contra x_ode, con leyenda)

Mirar el numero de la celda anterior antes de seguir: **la perdida sin la
ecuacion da cero**, y sin embargo la funcion esta mal.

Es el mismo aviso que atraviesa toda la practica. Una perdida chica solo dice
que el modelo cumple lo que le pedimos. Si le pedimos poco, lo cumple perfecto
y no sirve. El modelo aprende lo que la perdida paga: ni mas, ni menos.

## Extra para llevar: el autoencoder generativo (VAE)

No es parte de la practica: es un recurso para seguir en casa. Cambiar
`CORRER_VAE = True` y correr la celda (tarda menos de un minuto).

Nuestro autoencoder manda cada galaxia a **un punto** del espacio latente.
Sirve para comprimir y comparar, pero no para *inventar*: un punto cualquiera
del espacio, decodificado, suele ser basura. El **VAE** cambia eso: el encoder
devuelve una gaussiana (media $\mu$ y desvio $\sigma$) en vez de un punto, y
la perdida suma un termino que empuja todas esas gaussianas hacia la normal
estandar. El espacio queda lleno, sin huecos, y entonces sortear un $z$ al
azar y decodificarlo produce **galaxias nuevas**, que es lo que hace la
ultima figura.

Un detalle del codigo, por si lo leen: no se puede derivar a traves de un
sorteo, asi que el sorteo se hace aparte ($\varepsilon$) y se escribe
$z = \mu + \sigma\varepsilon$, que si es derivable. Se llama
*reparametrizacion*.

In [ ]:
CORRER_VAE = False        # ponerlo en True para correr esta parte

if CORRER_VAE:
    class VAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.tronco = nn.Sequential(
                nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
                nn.Flatten(),
            )
            self.a_mu     = nn.Linear(64 * 4 * 4, DIM_LATENTE)
            self.a_logvar = nn.Linear(64 * 4 * 4, DIM_LATENTE)
            self.expandir = nn.Linear(DIM_LATENTE, 64 * 4 * 4)
            self.decoder = nn.Sequential(
                nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
                nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
                nn.ConvTranspose2d(16, 3, 4, stride=2, padding=1), nn.Sigmoid(),
            )

        def decodificar(self, z):
            return self.decoder(self.expandir(z).view(-1, 64, 4, 4))

        def forward(self, x):
            h = self.tronco(x)
            mu, logvar = self.a_mu(h), self.a_logvar(h)
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)   # reparametrizacion
            return self.decodificar(z), mu, logvar

    BETA = 1e-3          # cuanto pesa el termino que ordena el espacio
    vae = VAE()
    opt = torch.optim.Adam(vae.parameters(), lr=2e-3)
    for ep in range(EPOCAS):
        perm = torch.randperm(len(X_train))
        for i in range(0, len(X_train), 128):
            xb = X_train[perm[i:i + 128]]
            opt.zero_grad()
            xhat, mu, logvar = vae(xb)
            rec = ((xhat - xb) ** 2).mean()
            kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(1)).mean()
            (rec + BETA * kl).backward()
            opt.step()
        if (ep + 1) % 5 == 0:
            print(f"  epoca {ep + 1:2d}   rec {float(rec):.5f}   KL {float(kl):.2f}")

    # Lo que el autoencoder normal no puede hacer: inventar galaxias.
    with torch.no_grad():
        nuevas = vae.decodificar(torch.randn(8, DIM_LATENTE))
    fig, axs = plt.subplots(1, 8, figsize=(9, 1.5))
    for j in range(8):
        axs[j].imshow(nuevas[j].permute(1, 2, 0).clamp(0, 1))
        axs[j].set_xticks([]); axs[j].set_yticks([])
    fig.suptitle("galaxias sorteadas del VAE (ningun dato de entrada)")
    plt.tight_layout(); plt.show()
else:
    print("VAE desactivado. Para correrlo, poner CORRER_VAE = True.")

## Extra: los mismos datos desde Multimodal Universe

[Multimodal Universe](https://huggingface.co/MultimodalUniverse) es un catalogo
de ~100 TB de datos astronomicos (imagenes, espectros, curvas de luz, catalogos)
publicados en un formato comun y accesibles en *streaming*: se van bajando los
lotes a medida que se piden, sin descargar el dataset entero.

Para la clase usamos el archivo local, que no depende de la conexion. Esta celda
esta para probar en casa. Cambiar `USAR_MMU = True` para correrla.

In [ ]:
USAR_MMU = False     # ponerlo en True para probar el streaming

if USAR_MMU:
    # pip install datasets
    from datasets import load_dataset

    ds = load_dataset("MultimodalUniverse/galaxy10_decals",
                      split="train", streaming=True)
    lote = list(ds.take(8))
    print("campos disponibles:", list(lote[0].keys()))

    fig, axs = plt.subplots(1, 8, figsize=(9, 1.6))
    for ax, ej in zip(axs, lote):
        img = np.asarray(ej["image"] if "image" in ej else ej["rgb_image"])
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Multimodal Universe, en streaming")
    plt.tight_layout(); plt.show()
else:
    print("Streaming desactivado. Para probarlo, poner USAR_MMU = True.")

## Resumen final

Correrla al final: junta en un solo lugar todos los numeros de la practica.

In [ ]:
def _v(n):
    "El valor si existe; None si esa parte no se completo."
    return globals().get(n, None)

print("PRACTICA 4 - RESUMEN FINAL")
print(f"datos: {MODO}")
print(f"dimension latente: {DIM_LATENTE}   epocas: {EPOCAS}")
print(f"galaxias: {len(X_train)} entrenamiento / {len(X_val)} validacion")
print()
print(f"{'modelo':<30}{'desalineo':>11}{'sonda':>9}")
filas = [("autoencoder base", "des_base", "sonda_base"),
         ("invariante por construccion", "des_inv", "sonda_inv"),
         ("contrastivo", "des_con", "sonda_con")]
for nombre, a, c in filas:
    d = _v(a)
    if d is None:
        print(f"{nombre:<30}{'(sin completar)':>20}")
        continue
    v = _v(c)
    st = f"{v:>9.3f}" if v is not None else f"{'-':>9}"
    print(f"{nombre:<30}{d:>11.3f}{st}")
print()
print("azar = 0.333;  desalineo 0 = invariancia perfecta")
print()
if _v("techo") is not None:
    print(f"techo supervisado (todas las etiquetas): {techo:.3f}")
else:
    print("techo supervisado: (sin completar)")
print()
if _v("u_sin") is not None:
    print("PINN (Parte 8): error maximo con la ecuacion "
          f"{np.abs(u_pinn - u_exacta).max():.4f}, sin la ecuacion "
          f"{np.abs(u_sin - u_exacta).max():.4f}")
else:
    print("PINN (Parte 8): (sin completar)")